In [1]:
import sys
import platform
from os.path import join, exists, abspath, dirname
from os import getcwd, makedirs, remove
from glob import glob

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colormaps, cm
from matplotlib import colors as mcolors
import scipy
from scipy.stats import kendalltau
from scipy.stats import ttest_1samp, pearsonr, ttest_rel
from scipy.io import loadmat
from sklearn.metrics import r2_score
from statsmodels.stats.anova import AnovaRM
import seaborn as sns
import h5py

from tqdm import tqdm

In [2]:
import nibabel as nb
from nilearn import plotting, image
from nipype.interfaces import fsl

In [3]:
from nilearn.mass_univariate import permuted_ols

In [4]:
dir_current = getcwd().replace('\\','/')

tmp = dir_current.split('/')
idx = [ii for ii, s in enumerate(tmp) if s=='github'][0]

dir_git = '/'.join(tmp[:idx+1])
dir_git

'/Users/sungbeenpark/github'

In [5]:
dname = join(dir_git,'nitools')
sys.path.append(dname)
import nitools as nt

In [6]:
dname = join(dir_git,'SUITPy')
sys.path.append(dname)
import SUITPy as suit

In [7]:
dname = join(dir_git)
sys.path.append(dname)
import surfAnalysisPy as surf

In [8]:
dname = join(dir_git,'Functional_Fusion')
sys.path.append(dname)
import Functional_Fusion.atlas_map as am
import Functional_Fusion.reliability as rel

In [9]:
dname = join(dir_git,'AnatSearchlight')
sys.path.append(dname)
import AnatSearchlight.searchlight as sl

In [10]:
dname = abspath(join(dir_git,'PcmPy'))
sys.path.append(dname)
import PcmPy as pcm

In [11]:
dname = join(dir_git,'SeqSpatialSupp_fMRI')
sys.path.append(dname)
from SSS import deal_spm
from SSS import util as su
from SSS import stat as sstat
from SSS import plot as splt
from SSS import image as simage
from SSS import glmsingle as ssingle

---

In [12]:
hem = 'L'

In [13]:
list_sn = su.get_list_sn()

---

### surface adjacency

In [14]:
fname = join(dir_git,'fs_LR_32/fs_LR.32k.L.midthickness.surf.gii')
exists(fname)

True

In [15]:
gii = nb.load(fname)
coords = gii.darrays[0].data
faces = gii.darrays[1].data
print(coords.shape, faces.shape)

(32492, 3) (64980, 3)


In [16]:
V = coords.shape[0]
adj = [[] for _ in range(V)]
for f in faces:
    a, b, c = f
    adj[a].extend([b,c])
    adj[b].extend([a,c])
    adj[c].extend([a,b])

---

In [17]:
dir_surf = join(ssingle.get_dir_glmsingle(),'surfaceWB')
dir_surf

'/Volumes/Diedrichsen_data$/data/SeqSpatialSupp_fMRI/GLMsingle/surfaceWB'

In [18]:
dir_group = join(dir_surf,'group')

In [19]:
glm = 1
dir_work = join(dir_surf,'glm_%1d'%glm)
dir_work

'/Volumes/Diedrichsen_data$/data/SeqSpatialSupp_fMRI/GLMsingle/surfaceWB/glm_1'

---

### Load individual dataset

In [20]:
model = 'first_finger'

In [21]:
fname = join(dir_work,'cifti.%s.glm_%1d.searchlight.mean_dist.%s.dscalar.nii'%(hem,glm,model))
dataset = nb.load(fname)
N = dataset.shape[0]
dataset.shape

(12, 32492)

---

### sign-flip permutation

In [25]:
thresh = sstat.convert_alpha_to_tval(alpha=0.05, df=N-1, alternative='greater')
thresh

np.float64(2.200985160082949)

In [26]:
X = dataset.get_fdata()
n_perm = 5000
max_cluster_masses = []

for _ in tqdm(range(n_perm)):
    signs = np.random.choice([1, -1], size=(12, 1))
    X_perm = X * signs

    t_perm, _ = ttest_1samp(X_perm, popmean=0, axis=0, alternative='greater')

    supra_perm = t_perm > thresh

    # cluster detection (same as above)
    visited = np.zeros(V, dtype=bool)
    max_mass = 0

    for v in np.where(supra_perm)[0]:
        if visited[v]:
            continue
        stack = [v]
        cluster = []
        while stack:
            u = stack.pop()
            if visited[u] or not supra_perm[u]:
                continue
            visited[u] = True
            cluster.append(u)
            for w in adj[u]:
                if not visited[w] and supra_perm[u]:
                    stack.append(w)

        if len(cluster) > 0:
            mass = np.sum(t_perm[cluster])
            max_mass = max(max_mass, mass)

    max_cluster_masses.append(max_mass)

100%|████████████████████████████████████| 5000/5000 [33:19:03<00:00, 23.99s/it]


In [27]:
max_cluster_masses = np.array(max_cluster_masses)
np.save(join(dir_work,'max_cluster_masses'), max_cluster_masses)

---

### Check the result

In [28]:
border, brdr = simage.get_border(dir_git=dir_git, atlas='sulcus')

In [ ]:
def get_RDM(model, dir_work, alpha=0.05, alternative='greater'):
    fname = join(dir_work,'cifti.%s.glm_%1d.searchlight.mean_dist.%s.dscalar.nii'%(hem,glm,model))
    data = np.array(nb.load(fname).get_fdata())
    ds = data[0].copy()

    thresh = sstat.convert_alpha_to_tval(alpha=alpha, df=len(list_sn)-1, alternative=alternative)
    if alternative=='greater':
        idx = data[1] >= thresh
    elif alternative=='lower':
        idx = data[1] <= thresh
    elif alternative=='two-tailed':
        idx = abs(data[1]) > thresh
    ds[~idx] = np.nan

    return ds

In [ ]:
ds = get_RDM(model=model, dir_work=dir_group, alpha=0.05, alternative='greater')

In [ ]:
fig, ax = plt.subplots()
plt.sca(ax)
g = surf.plot.plotmap(
    data=ds,
    surf='fs32k_%s'%hem,
    alpha=1,
    cmap=cm.jet, colorbar=True,
    # cscale=[0.0019, 0.23],
    # threshold=thresh,
    borders=border, bordercolor='black', bordersize=1,
    overlay_type='func', render='matplotlib',
)

In [ ]:
# fig.savefig(
#     join(su.get_dir_root(),'RDM_flat.%s.png'%model),
#     dpi=300, facecolor=[1,1,1,1], bbox_inches = "tight"
# )

---
---

---